# Meenakshi Polymers – OCR for every form template (Qwen3-VL, Colab GPU)

Feed any scanned PDF → for every page you get the **expected JSON** (same shapes as `OCR_JSON_SCHEMA.md`).

| Template id | Heading that identifies it | Kind | Data block |
|---|---|---|---|
| `SUPPLIER_SEGREGATION` | `… SEGREGATION REPORT` | record | `entries[]` |
| `SUPPLIER_REWORK` | `… REWORK REPORT` | record | `entries[]` |
| `SUPPLIER_REJECTION` | `… REJECTION REPORT` | record | `entries[]` |
| `SUPPLIER_LOT_REJECTION_SUMMARY` | `… LOT REJECTION SUMMARY` | record | `entries[]` |
| `SCRAP_NOTE` | `SCRAP NOTE` | record | `entries[]` |
| `LINE_REJECTION` | `LINE REJECTION REPORT` (+ shop: PRESS SHOP / WELD SHOP) | record | `entries[]` |
| `ST_PASS_MONITORING` | `… COMPONENT STRAIGHT PASS MONITORING …` | straight-pass grid | `parts[].days{}` |
| `MIS` (one template, 4 sub-types) | `DAILY MIS REPORT` / `FI VISUAL INSPECTION` | defect matrix | `sections[].rows[]` |

### How the template is decided (never from the file name or folder)
1. **Header read** – Qwen transcribes the printed heading lines and header fields from the top of the page.
2. **Rules** – plain keyword rules on those lines, most specific first (`LOT REJECTION SUMMARY` before `REJECTION REPORT`,
   `LINE REJECTION REPORT` before `REJECTION REPORT`). Small reading errors are tolerated (fuzzy match).
3. **Input override** – `TEMPLATE_OVERRIDE` forces a template for matching files; `STRICT_INPUT = True` uses only the override.
   Without a GPU (`DRY_RUN`) the override (or a PDF text layer, if the scan has one) is the only source.

### MIS: one template, sub-types from the header
All four MIS sheets share one skeleton: `SR.NO | DEFECT | [REASON] | part columns … | TOTAL | [disposition columns … | TOTAL]`,
printed defect rows grouped into sections, each closed by a TOTAL row (and a %/straight-pass row), then summary rows.
The sub-type only supplies **stable column keys** and a check list:

| Sub-type | Detected by |
|---|---|
| `FI_VISUAL_POWDER_COATING` | title contains `FI VISUAL INSPECTION` |
| `PLATING_DAILY` | plating columns (`RENICKEL`, `RECHROME`, `CR AAEL` …) or doc no `…/PL` |
| `POWDER_COATING_DAILY_QA` | powder-coating sheet **with** disposition columns (`REWORK`, `OK`, `B GRADE`, `SCRAP`) |
| `POWDER_COATING_DAILY` | powder-coating sheet without them |

**Two-level column headers** (e.g. `FOR REWORK` over `MINOR | MAJOR`, `REJECTION` over `SCRAP`, while `TOTAL` spans both levels):
1. The header strip that is stacked on top of every band image covers the **whole header block** (both levels),
   so the model always sees `FOR REWORK / MINOR`.
2. A dedicated call reads the header as **leaf columns left→right with their group**:
   `{"label": "MINOR", "group": "FOR REWORK"}`. A group heading is never a column itself; a column that spans both
   levels has `group: null`.
3. Keys: the sub-type's canonical key when the label aligns (`for_rework_minor`, `rejection_scrap`), otherwise
   `slug(group + label)`. Duplicate labels become unique (`total` → second one `grand_total`).
4. The JSON keeps `columns` (keys in order), `column_meta` (`key, label, group, role`) and `column_groups`.
5. Validation uses the roles: `sum(part columns) == TOTAL`, `sum(disposition columns) == grand_total` (or `TOTAL`).

### Debugging in Colab
Every page writes `debug/<pdf>/p<N>/`: numbered preprocessing images, the **exact image of every model call**
(`model_inputs/`), the image at the resolution Qwen used (`model_view/`) and `model_io.jsonl`
(prompt, raw answer, parsed JSON, tokens, seconds). `VERBOSE` prints every call, `SHOW_INLINE` shows overlays
and crops in the notebook, `inspect_page()` re-displays any page afterwards, `report.html` puts it all side by side.

## 1. Install

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Qwen3-VL needs transformers >= 4.57
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.57.0", "accelerate", "pymupdf", "opencv-python-headless", "openpyxl"],
                   check=True)
print("Running in Colab:", IN_COLAB)

In [ ]:
import os, re, json, time, shutil, html, fnmatch, difflib, traceback
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import pymupdf
import matplotlib
import matplotlib.pyplot as plt

try:
    from IPython.display import display, HTML
except ImportError:                       # plain python (local testing)
    display, HTML = print, str

try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
    print("torch", torch.__version__, "| CUDA:", HAS_CUDA)
    if HAS_CUDA:
        p = torch.cuda.get_device_properties(0)
        print(f"GPU: {p.name}  {p.total_memory / 1e9:.1f} GB")
except ImportError:
    torch, HAS_CUDA = None, False
    print("torch not installed")
print("opencv", cv2.__version__, "| numpy", np.__version__, "| pymupdf", pymupdf.__version__)

## 2. Configuration

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/MeenakshiPublic"

def mount_drive():
    if not IN_COLAB:
        return
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")

mount_drive()

# files or folders (searched recursively); the folder layout is NOT used to pick the template
INPUT_PATHS = ([f"{DRIVE_ROOT}/Meenakshi ALL Documents"] if IN_COLAB
               else [os.environ.get("MP_INPUT", "Dataset/Meenakshi ALL Documents")])
OUTPUT_DIR  = Path(f"{DRIVE_ROOT}/ocr_all_forms_output") if IN_COLAB else Path(os.environ.get("MP_OUTPUT", "ocr_all_forms_output"))
ONLY_FILES  = []          # substrings of the path to process only some files, e.g. ["SCRAP", "mis plating 01"]
MAX_PAGES_PER_PDF = None  # e.g. 1 for a quick look

# ---- template input (optional) ------------------------------------------------------------------
# glob on the file name or on the path relative to the input folder -> template id.
# "MIS" lets the sub-type be detected; "MIS:PLATING_DAILY" forces it too.
TEMPLATE_OVERRIDE = {}    # e.g. {"scan_0412.pdf": "SCRAP_NOTE", "*plating*": "MIS:PLATING_DAILY"}
STRICT_INPUT = False      # True = only TEMPLATE_OVERRIDE decides (pages without an override are skipped)
if not IN_COLAB and os.environ.get("MP_OVERRIDE"):
    TEMPLATE_OVERRIDE = json.loads(os.environ["MP_OVERRIDE"])
if not IN_COLAB and os.environ.get("MP_ONLY"):
    ONLY_FILES = os.environ["MP_ONLY"].split(",")

# ---- model ---------------------------------------------------------------------------------------
MODEL_ID       = "Qwen/Qwen3-VL-4B-Instruct"   # "Qwen/Qwen3-VL-8B-Instruct" on L4/A100 for better handwriting
DRY_RUN        = False    # True = preprocessing + saved model inputs only, no model
MAX_NEW_TOKENS = 2048

# ---- debugging -----------------------------------------------------------------------------------
VERBOSE         = True    # one line per model call (+ start of the answer)
SHOW_INLINE     = True    # show overlays and crops in the notebook while running
SHOW_MAX_IMAGES = 4       # inline crops per page
SAVE_MODEL_VIEW = True    # also save every input at the resolution Qwen processed it
INCLUDE_META_IN_JSON = True   # "_meta" (template decision, validation issues, debug paths) in the JSON files

# ---- preprocessing (pixel values at 2480 px page width ≈ A4 at 300 dpi) -------------------------
PORTRAIT_WIDTH, LANDSCAPE_WIDTH = 2480, 3508
HEADER_BAND_FRAC = {"portrait": 0.24, "landscape": 0.14}   # top part of the page read for the template decision
MIN_ROW_INK          = 1500   # record forms: handwriting pixels for a ruled row to count as filled
CONTINUATION_MAX_INK = 150    # record forms: less ink in the first column -> row continues the previous entry
TOP_PAD, BOTTOM_PAD, SIDE_PAD, IMAGE_MARGIN = 10, 22, 40, 20
MIS_BAND_PX       = 300       # MIS: height of one band of rows sent to the model (~6 rows)
MIS_BAND_OVERLAP  = 60        # MIS: overlap between bands; rows seen twice are de-duplicated
MIS_SNAP_PX       = 25        # MIS: move a band cut onto a printed row line when one is this close
ST_MIN_DAY_INK    = 400       # straight-pass: ink needed in a part×day cell group to send it to the model

if not DRY_RUN and not HAS_CUDA:
    print("⚠ No GPU found -> DRY_RUN (preprocessing only). Colab: Runtime → Change runtime type → T4 GPU")
    DRY_RUN = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("input :", INPUT_PATHS)
print("output:", OUTPUT_DIR.resolve())
print("DRY_RUN:", DRY_RUN, "| override:", TEMPLATE_OVERRIDE or "-", "| strict:", STRICT_INPUT)

### PDFs to process

In [ ]:
def list_pdfs(paths):
    found = []
    for p in map(Path, paths):
        if p.is_file() and p.suffix.lower() == ".pdf":
            found.append((p, p.name))
        elif p.is_dir():
            for f in sorted(p.rglob("*")):
                if f.is_file() and f.suffix.lower() == ".pdf":
                    found.append((f, f.relative_to(p).as_posix()))
        else:
            print("⚠ not found:", p)
    if ONLY_FILES:
        found = [(f, r) for f, r in found if any(s.upper() in r.upper() for s in ONLY_FILES)]
    return found

PDFS = list_pdfs(INPUT_PATHS)
print(f"{len(PDFS)} PDF(s)")
for f, rel in PDFS:
    print("  ", rel)

## 3. Templates
Structure only (no pixel coordinates): which header fields and signatures exist, the record fields with their printed
column header and type, how many printed rows sit above the data (record forms), and validation rules.

In [ ]:
def S(header): return (header, "string")
def I(header): return (header, "integer")

SUPPLIER_SIGNS = ["incoming_qa", "purchase", "hod_qa", "hod_plant"]

TEMPLATES = {
    "SUPPLIER_LOT_REJECTION_SUMMARY": dict(
        kind="record", title="Supplier Lot Rejection Summary", match=["LOT REJECTION SUMMARY", "LOT REJECTION"],
        header_rows=3, footer_rows=1, expected_col_lines=13, first_col="date",
        header_keys=["company", "title", "month"], signature_keys=SUPPLIER_SIGNS,
        fields={"date": S("DATE"), "part_name": S("PART NAME"), "supplier_name": S("SUPPLIER NAME"),
                "lot_qty": I("LOT QTY"), "rework_qty": I("REW QTY"), "rejection_qty": I("REJ QTY"),
                "ok_qty": I("OK QTY"), "problem_description": S("PROBLEM DESCRIPTION"), "action": S("ACTION"),
                "return_status": S("RETURN STATUS"), "capa_status": S("CAPA STATUS"), "remarks": S("REMARKS")},
        required=["date", "part_name", "lot_qty", "ok_qty"],
        rules=[("lot_qty == rejection_qty + ok_qty", ["lot_qty", "rejection_qty", "ok_qty"],
                lambda r: r["lot_qty"] == r["rejection_qty"] + r["ok_qty"])]),
    "SUPPLIER_SEGREGATION": dict(
        kind="record", title="Supplier Segregation Report", match=["SEGREGATION REPORT"],
        header_rows=3, footer_rows=1, expected_col_lines=9, first_col="date",
        header_keys=["company", "title", "month"], signature_keys=SUPPLIER_SIGNS,
        fields={"date": S("DATE"), "part_name": S("PART NAME"), "supplier_name": S("SUPPLIER NAME"),
                "lot_qty": I("LOT QTY"), "segregation_qty": I("SEGREGATION QTY"), "ok_qty": I("OK QTY"),
                "problem_description": S("PROBLEM DESCRIPTION"), "remarks": S("REMARKS")},
        required=["date", "part_name", "lot_qty", "segregation_qty", "ok_qty"],
        rules=[("lot_qty == segregation_qty + ok_qty", ["lot_qty", "segregation_qty", "ok_qty"],
                lambda r: r["lot_qty"] == r["segregation_qty"] + r["ok_qty"])]),
    "SUPPLIER_REWORK": dict(
        kind="record", title="Supplier Rework Report", match=["REWORK REPORT"],
        header_rows=3, footer_rows=1, expected_col_lines=9, first_col="date",
        header_keys=["company", "title", "month"], signature_keys=SUPPLIER_SIGNS,
        fields={"date": S("DATE"), "part_name": S("PART NAME"), "supplier_name": S("SUPPLIER NAME"),
                "lot_qty": I("LOT QTY"), "rework_qty": I("REWORK QTY"), "ok_qty": I("OK QTY"),
                "problem_description": S("PROBLEM DESCRIPTION"), "remarks": S("REMARKS")},
        required=["date", "part_name", "lot_qty", "rework_qty", "ok_qty"],
        rules=[("rework_qty <= lot_qty", ["rework_qty", "lot_qty"], lambda r: r["rework_qty"] <= r["lot_qty"]),
               ("ok_qty <= lot_qty", ["ok_qty", "lot_qty"], lambda r: r["ok_qty"] <= r["lot_qty"])]),
    "SUPPLIER_REJECTION": dict(
        kind="record", title="Supplier Rejection Report", match=["SUPPLIER REJECTION REPORT", "REJECTION REPORT"],
        header_rows=3, footer_rows=1, expected_col_lines=9, first_col="date",
        header_keys=["company", "title", "month"], signature_keys=SUPPLIER_SIGNS,
        fields={"date": S("DATE"), "part_name": S("PART NAME"), "supplier_name": S("SUPPLIER NAME"),
                "lot_qty": I("LOT QTY"), "rejection_qty": I("REJECTION QTY"), "ok_qty": I("OK QTY"),
                "problem_description": S("PROBLEM DESCRIPTION"), "remarks": S("REMARKS")},
        required=["date", "part_name", "lot_qty", "rejection_qty", "ok_qty"],
        rules=[("lot_qty == rejection_qty + ok_qty", ["lot_qty", "rejection_qty", "ok_qty"],
                lambda r: r["lot_qty"] == r["rejection_qty"] + r["ok_qty"])]),
    "LINE_REJECTION": dict(
        kind="record", title="Line Rejection Report", match=["LINE REJECTION REPORT"],
        header_rows=5, footer_rows=1, expected_col_lines=8, first_col="s_no",
        header_keys=["company", "title", "doc_no", "rev_no", "rev_date", "org_date", "shop", "month"],
        signature_keys=["prepared_by", "prod_head", "qa_head", "vp_operations"],
        fields={"s_no": I("S.NO."), "item_code": S("ITEM CODE"), "material_particulars": S("MATERIAL PARTICULARS"),
                "unit": S("UNIT"), "rejection_qty": I("REJECTION QTY"), "vendor_name": S("VENDOR NAME"),
                "reason_remarks": S("REASON / REMARKS")},
        required=["item_code", "rejection_qty"],
        rules=[("item_code looks like SF500257 / RM500003", ["item_code"],
                lambda r: re.fullmatch(r"[A-Z]{2,3}\d{6}", r["item_code"].replace(" ", "").upper()) is not None)]),
    "SCRAP_NOTE": dict(
        kind="record", title="Scrap Note", match=["SCRAP NOTE"],
        header_rows=4, footer_rows=1, expected_col_lines=8, first_col="s_no",
        header_keys=["company", "title", "doc_no", "rev_no", "rev_date", "org_date", "month", "period"],
        signature_keys=["prepared_by", "qc_head", "plating_head", "vp_operation"],
        fields={"s_no": I("S.No."), "item_code": S("ITEM CODE"), "item_name": S("ITEM NAME"), "unit": S("UNIT"),
                "rejection_qty": I("REJECTION QTY."), "vendor_name": S("VENDOR. NAME"),
                "reason_remarks": S("REASON / REMARKS")},
        required=["item_code", "rejection_qty"],
        rules=[("item_code looks like WLD602503", ["item_code"],
                lambda r: re.fullmatch(r"WLD\d{6}", r["item_code"].replace(" ", "").upper()) is not None)]),
    "ST_PASS_MONITORING": dict(
        kind="stpass", title="BS6 Model Component Straight Pass Monitoring Sheet",
        match=["COMPONENT STRAIGHT PASS MONITORING", "STRAIGHT PASS MONITORING"],
        header_keys=["company", "title", "month"], header_bands=2,
        params=[("checked_qty", "CHECKED QTY."), ("ok_qty", "OK QTY."), ("rej_qty", "REJ. QTY."),
                ("rew_qty", "REW. QTY."), ("total_rework_ok_qty", "TOTAL REWORK OK QTY"),
                ("inspection_chipping_pending", "INSPECTION/CHIPPING PENDING"),
                ("visual_defects_straight_pass_pct", "VISUAL DEFECTS STRAIGHT PASS (%)"),
                ("non_conformity_count", "NO. OF NON CONFORMITY")],
        shifts=["A", "B", "TOTAL"], days_per_sheet=10),
    "MIS": dict(
        kind="matrix", title="Daily MIS defect report",
        match=["DAILY MIS REPORT", "FI VISUAL INSPECTION", "MIS REPORT"],
        header_keys=["company", "title", "doc_no", "rev_date", "org_date", "rev_no", "report", "shift", "date", "month"],
        signature_keys=["prepared_by", "checked_by", "reviewed_by", "approved_by"]),
}
# rules are tried in this order: most specific heading first
MATCH_ORDER = ["ST_PASS_MONITORING", "MIS", "SUPPLIER_LOT_REJECTION_SUMMARY", "LINE_REJECTION",
               "SUPPLIER_SEGREGATION", "SUPPLIER_REWORK", "SUPPLIER_REJECTION", "SCRAP_NOTE"]

# ---- MIS sub-types: canonical columns (printed label, key, group) left to right ------------------
def C(label, key, group=None): return dict(label=label, key=key, group=group)

PC_PARTS = [C("SG-AAH", "sg_aah"), C("SG-AAE", "sg_aae"), C("SW (AAH)", "sw_aah"), C("SP-AAE", "sp_aae"),
            C("SG (AACY)", "sg_aacy"), C("SW (AACY)", "sw_aacy"), C("EG-KVH", "eg_kvh"),
            C("CROSS PIPE-AAEL", "cross_pipe_aael"), C("HANDLE ABZA", "handle_abza"), C("EG-ABZA", "eg_abza"),
            C("PB-ABZ", "pb_abz"), C("PGC-ACK", "pgc_ack"), C("PB-ACK", "pb_ack"), C("HANDLE AACK", "handle_aack"),
            C("SIDE STAND", "side_stand"), C("SG-AAEL", "sg_aael")]
MIS_LABEL_COLS = [C("SR.NO", "sr_no"), C("DEFECT", "defect"), C("REASON", "reason")]

MIS_VARIANTS = {
    "FI_VISUAL_POWDER_COATING": dict(
        title="FI Visual Inspection 100% (Powder Coating)",
        columns=[C("S.NO.", "sr_no"), C("DESCRIPTION OF DEFECT", "defect"),
                 C("SARI GUARD-AAH", "sari_guard_aah"), C("SARI GUARD-AAE", "sari_guard_aae"),
                 C("STEP WOMEN-AAH", "step_women_aah"), C("STEP PILLION-AAE", "step_pillion_aae"),
                 C("SARI GUARD-AACY", "sari_guard_aacy"), C("STEP WOMEN AACY", "step_women_aacy"),
                 C("ENGINE GUARD-KVH", "engine_guard_kvh"), C("CROSS PIPE AAEL", "cross_pipe_aael"),
                 C("HANDEL ABZA", "handle_abza"), C("ENGINE GUARD-ABZA PAINTED", "engine_guard_abza_painted"),
                 C("PEDAL BRAKE-ABZA", "pedal_brake_abza"), C("SIDE STAND ABZA", "side_stand_abza"),
                 C("TOTAL", "total"), C("MINOR", "for_rework_minor", "FOR REWORK"),
                 C("MAJOR", "for_rework_major", "FOR REWORK"), C("SCRAP", "rejection_scrap", "REJECTION"),
                 C("B-GRADE/GP", "b_grade_gp")]),
    "PLATING_DAILY": dict(
        title="Daily MIS Report - Plating",
        columns=MIS_LABEL_COLS + [C("CR AAEL", "cr_aael"), C("CR AAD", "cr_aad"), C("EG-KST", "engine_guard"),
                                  C("PB-AAE", "pedal_brake"), C("PGC", "pgc"), C("HANDLE", "handle"),
                                  C("TOTAL", "total"), C("RENICKEL", "renickel"), C("RECHROME", "rechrome"),
                                  C("BUFF", "buff"), C("OK", "ok"), C("B GRADE", "b_grade"), C("SCRAP", "scrap"),
                                  C("TOTAL", "grand_total")]),
    "POWDER_COATING_DAILY_QA": dict(
        title="Daily MIS Report - Powder Coating (with disposition)",
        columns=MIS_LABEL_COLS + PC_PARTS + [C("TOTAL", "total"), C("REWORK", "rework"),
                                             C("MAJOR REWORK", "major_rework"), C("OK", "ok"),
                                             C("B GRADE", "b_grade"), C("SCRAP", "scrap"), C("TOTAL", "grand_total")]),
    "POWDER_COATING_DAILY": dict(
        title="Daily MIS Report - Powder Coating",
        columns=MIS_LABEL_COLS + PC_PARTS + [C("TOTAL", "total")]),
}
PLATING_MARKERS = ["RENICKEL", "RECHROME", "CR AAEL", "CR AAD", "EG-KST"]
DISPOSITION_MARKERS = ["REWORK", "MAJOR REWORK", "OK", "B GRADE", "SCRAP"]

print("templates:", ", ".join(TEMPLATES), "| MIS sub-types:", ", ".join(MIS_VARIANTS))

## 4. Image preprocessing and table geometry

In [ ]:
class Debug:
    """Numbered debug images for one page."""
    def __init__(self, root):
        self.root = Path(root); self.root.mkdir(parents=True, exist_ok=True); self.n = 0
    def save(self, name, img):
        path = self.root / f"{self.n:02d}_{name}.png"; self.n += 1
        cv2.imwrite(str(path), img)
        return path

def show(img, title="", width=14):
    if img is None:
        return
    h, w = img.shape[:2]
    plt.figure(figsize=(width, max(1.0, width * h / w)))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img, cmap="gray")
    plt.title(title, fontsize=10); plt.axis("off"); plt.show(); plt.close()

def load_page(page):
    # rendering (not extracting the JPEG) applies the PDF placement matrix: un-flips upside-down / sideways scans
    infos = page.get_image_info()
    if infos:
        info = max(infos, key=lambda i: i["width"] * i["height"])
        zoom = info["width"] / max(info["bbox"][2] - info["bbox"][0], 1)
    else:
        zoom = 300 / 72
    zoom = min(max(zoom, 150 / 72), 400 / 72)
    pix = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom), colorspace=pymupdf.csRGB, alpha=False)
    rgb = np.frombuffer(pix.samples, np.uint8).reshape(pix.h, pix.stride)[:, :pix.w * 3].reshape(pix.h, pix.w, 3)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), dict(zoom=round(zoom, 3), scanned=bool(infos))

def normalise_width(img):
    width = PORTRAIT_WIDTH if img.shape[0] >= img.shape[1] else LANDSCAPE_WIDTH
    s = width / img.shape[1]
    return img if abs(s - 1) < 0.01 else cv2.resize(img, None, fx=s, fy=s,
                                                     interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)

def to_gray(img): return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def binarize(gray):  # ink = white
    return cv2.adaptiveThreshold(~gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, -2)

def horizontal_lines(bw):
    return cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (bw.shape[1] // 25, 1)))

def vertical_lines(bw):
    return cv2.morphologyEx(bw, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, bw.shape[0] // 40)))

def deskew(img):
    small = cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)    # 4x faster, same angle
    h = horizontal_lines(binarize(to_gray(small)))
    segs = cv2.HoughLinesP(h, 1, np.pi / 1800, 100, minLineLength=small.shape[1] // 4, maxLineGap=10)
    if segs is None:
        return img, 0.0
    angles = [np.degrees(np.arctan2(y2 - y1, x2 - x1)) for x1, y1, x2, y2 in np.asarray(segs).reshape(-1, 4)]
    angles = [a for a in angles if abs(a) < 5]
    angle = float(np.median(angles)) if angles else 0.0
    M = cv2.getRotationMatrix2D((img.shape[1] / 2, img.shape[0] / 2), angle, 1)
    return cv2.warpAffine(img, M, img.shape[1::-1], flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255)), angle

def flatten_lighting(img):
    bg = cv2.morphologyEx(img, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (51, 51)))
    return cv2.divide(img, bg, scale=255)

def line_positions(mask, axis, frac):
    prof = mask.sum(axis=axis) / 255
    if prof.max() == 0:
        return []
    idx = np.where(prof > frac * prof.max())[0]
    groups = np.split(idx, np.where(np.diff(idx) > 5)[0] + 1)
    return [int(g.mean()) for g in groups if len(g)]

def filter_columns(gray, rows, cands):
    # a printed line is dark with equally bright paper on both sides; a shadow edge is bright on one side only
    kept, report = [], []
    strip = gray[rows[0]:rows[-1]] if len(rows) >= 2 else gray
    for k, x in enumerate(cands):
        line  = strip[:, max(x - 2, 0):x + 3].min(axis=1).mean()
        left  = strip[:, max(x - 15, 0):max(x - 8, 1)].mean()
        right = strip[:, x + 8:x + 15].mean() if x + 15 <= strip.shape[1] else 255.0
        border = k in (0, len(cands) - 1)
        ok = (min(left, right) - line > 25) and (border or abs(left - right) < 12)
        report.append(dict(x=x, line=round(float(line)), left=round(float(left)), right=round(float(right)), kept=bool(ok)))
        if ok:
            kept.append(x)
    return kept, report

def ink_mask(gray, h, v):
    ink = ((gray < 160) * 255).astype(np.uint8)
    ink = cv2.subtract(ink, cv2.dilate(h | v, np.ones((7, 7), np.uint8)))
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink)
    return (np.isin(lab, np.where(st[:, 4] >= 25)[0][1:]) * 255).astype(np.uint8)

def with_margin(img, m=IMAGE_MARGIN):
    return cv2.copyMakeBorder(img, m, m, m, m, cv2.BORDER_CONSTANT, value=(255, 255, 255))

def find_line_pieces(mask, min_frac=0.4, max_gap=80, min_fill=0.8):
    """Horizontal ruled lines as connected pieces (tolerant to slight tilt / page curl). A row of handwritten dashes
    is dropped because it is not continuous (fill < min_fill). Returns [{y, x0, x1}] sorted by y."""
    m = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_RECT, (31, 5)))
    n, lab, st, _ = cv2.connectedComponentsWithStats(m)
    pieces = []
    for i in range(1, n):
        x, y, w, h = st[i, :4]
        if w < 60:
            continue
        rows = (lab[y:y + h, x:x + w] == i).sum(axis=1)
        pieces.append(dict(y=int(y + np.argmax(rows)), x0=int(x), x1=int(x + w), lo=int(y), hi=int(y + h)))
    merged = []
    for p in sorted(pieces, key=lambda p: p["x0"]):
        for q in merged:
            if abs(q["y"] - p["y"]) <= 12 and p["x0"] - q["x1"] <= max_gap:
                q.update(x0=min(q["x0"], p["x0"]), x1=max(q["x1"], p["x1"]), lo=min(q["lo"], p["lo"]), hi=max(q["hi"], p["hi"]))
                break
        else:
            merged.append(dict(p))
    for q in merged:
        q["fill"] = float((mask[q["lo"]:q["hi"] + 1, q["x0"]:q["x1"]].max(axis=0) > 0).mean())
    merged = [q for q in merged if q["fill"] >= min_fill]
    if not merged:
        return []
    longest = max(q["x1"] - q["x0"] for q in merged)
    return sorted([q for q in merged if q["x1"] - q["x0"] >= min_frac * longest], key=lambda q: q["y"])

def uniform_run(pitches, tol=0.2, min_len=4):
    """Longest run of consecutive row heights within ±tol of the run's median -> (first, last) band index."""
    best = None
    for s in range(len(pitches)):
        e = s
        while e + 1 < len(pitches):
            ref = float(np.median(pitches[s:e + 2]))
            if all(abs(p - ref) <= tol * ref for p in pitches[s:e + 2]):
                e += 1
            else:
                break
        if e - s + 1 >= min_len and (best is None or e - s > best[1] - best[0]):
            best = (s, e)
    return best

def prepare_page(page, dbg):
    img, info = load_page(page);                 dbg.save("rendered", img)
    img = normalise_width(img)
    img, angle = deskew(img);                    dbg.save("deskewed", img)
    raw_gray = to_gray(img)                                   # before flattening: judges line contrast
    img = flatten_lighting(img);                 dbg.save("flattened", img)
    return img, raw_gray, angle, info

def basic_grid(img):
    gray = to_gray(img)
    bw = binarize(gray)
    h, v = horizontal_lines(bw), vertical_lines(bw)
    return gray, bw, h, v, line_positions(h, 1, 0.5), line_positions(v, 0, 0.3)

## 5. Model (Qwen3-VL) and call logging
Every call saves its exact input image, logs prompt + raw answer + parsed JSON + tokens + time to `model_io.jsonl`,
and (with `VERBOSE`) prints one line.

In [ ]:
model = processor = None
if not DRY_RUN:
    from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model = Qwen3VLForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, device_map="auto")
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model.eval()
    print("loaded", MODEL_ID, dtype, f"| GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
else:
    print("DRY_RUN: model not loaded")

In [ ]:
def parse_json(text):
    if not text or not text.strip():
        return None, "empty output"
    t = re.sub(r"```(?:json)?", "", text).strip()
    starts = [i for i in (t.find("{"), t.find("[")) if i >= 0]
    if not starts:
        return None, "no JSON in output"
    s = t[min(starts):max(t.rfind("}"), t.rfind("]")) + 1]
    err = None
    for cand in (s, re.sub(r",\s*([}\]])", r"\1", s)):
        try:
            return json.loads(cand), None
        except json.JSONDecodeError as e:
            err = e
    return None, f"invalid JSON ({err}) - output cut off? raise MAX_NEW_TOKENS"

def qwen_generate(image_path, prompt, max_new_tokens):
    messages = [{"role": "user", "content": [{"type": "image", "image": str(image_path)},
                                             {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new = out[:, inputs["input_ids"].shape[1]:]
    text = processor.batch_decode(new, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    grid = [int(x) for x in inputs["image_grid_thw"][0].tolist()]          # [t, h_patches, w_patches]
    patch = getattr(processor.image_processor, "patch_size", 16)
    return dict(raw=text, seconds=round(time.time() - t0, 2), input_tokens=int(inputs["input_ids"].shape[1]),
                output_tokens=int(new.shape[1]), grid_thw=grid, model_view_wh=(grid[2] * patch, grid[1] * patch),
                hit_token_limit=int(new.shape[1]) >= max_new_tokens)

class PageCtx:
    """Folders + call log for one page."""
    def __init__(self, pdf_rel, page_no, root):
        self.pdf_rel, self.page_no, self.dir = pdf_rel, page_no, Path(root)
        self.inputs = self.dir / "model_inputs"; self.inputs.mkdir(parents=True, exist_ok=True)
        self.view = self.dir / "model_view"
        self.dbg = Debug(self.dir / "debug")
        self.calls, self.shown = [], 0
        (self.dir / "model_io.jsonl").unlink(missing_ok=True)

def call_model(ctx, name, img, prompt, max_new_tokens=MAX_NEW_TOKENS):
    path = ctx.inputs / f"{len(ctx.calls):02d}_{name}.png"
    cv2.imwrite(str(path), img)
    if DRY_RUN:
        r = dict(raw="", seconds=0.0, input_tokens=None, output_tokens=None, grid_thw=None, model_view_wh=None,
                 hit_token_limit=False)
        parsed, err = None, "dry run"
    else:
        try:
            r = qwen_generate(path, prompt, max_new_tokens)
            parsed, err = parse_json(r["raw"])
        except Exception as e:                                   # keep going: one failed call must not stop the run
            r = dict(raw="", seconds=0.0, input_tokens=None, output_tokens=None, grid_thw=None, model_view_wh=None,
                     hit_token_limit=False)
            parsed, err = None, f"model error: {type(e).__name__}: {e}"
    view = None
    if SAVE_MODEL_VIEW and r.get("model_view_wh"):
        ctx.view.mkdir(parents=True, exist_ok=True)
        view = ctx.view / path.name
        cv2.imwrite(str(view), cv2.resize(img, tuple(r["model_view_wh"]), interpolation=cv2.INTER_AREA))
    rec = dict(name=name, image=str(path), image_wh=(img.shape[1], img.shape[0]), prompt=prompt, **r,
               parsed=parsed, error=err, model_view=str(view) if view else None)
    ctx.calls.append(rec)
    with open(ctx.dir / "model_io.jsonl", "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
    if VERBOSE:
        ans = re.sub(r"\s+", " ", r["raw"])[:110]
        print(f"      · {name:16s} img {img.shape[1]}x{img.shape[0]} view {r.get('model_view_wh')} "
              f"tok {r.get('input_tokens')}/{r.get('output_tokens')} {r['seconds']:5.1f}s "
              f"{'⚠ ' + err if err and not DRY_RUN else ''} {ans}")
    if SHOW_INLINE and ctx.shown < SHOW_MAX_IMAGES and name not in ("header", "mis_locate"):
        ctx.shown += 1
        show(img, f"{ctx.pdf_rel} p{ctx.page_no} · model input {path.name}", width=12)
    return parsed, rec

## 6. Template decision (header read + rules + override)

In [ ]:
HEADER_PROMPT = """This is the top part of a scanned factory form of Meenakshi Polymers.
Transcribe the printed heading and the header fields. Ignore the table rows below the column headers.
Return only JSON:
{"title_lines": [every printed title / heading line, top to bottom, exactly as printed],
 "company": printed company name or null,
 "doc_no": printed DOC NO value or null, "rev_no": REV NO value or null, "rev_date": REV DATE value or null,
 "org_date": ORG DATE / ORIGINAL DATE value or null,
 "month": handwritten month (after MONTH) or null, "date": handwritten date (after DATE) or null,
 "shift": shift letter if written or null,
 "report": handwritten text after "DAILY MIS REPORT-" or null,
 "shop": shop name printed below the title (e.g. "PRESS SHOP", "WELD SHOP") or null,
 "period": handwritten period / date range or null}
Copy text exactly as written. Do not guess values that are not on the page (use null)."""

def norm_text(s):
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z0-9%/]+", " ", str(s or "").upper())).strip()

def fuzzy_contains(text, phrase, thr=0.85):
    if phrase in text:
        return True
    tw, n = text.split(), len(phrase.split())
    for k in (n - 1, n, n + 1):                  # words may be split or joined ("WELDSHOP")
        for i in range(0, max(len(tw) - k + 1, 0)):
            if k > 0 and difflib.SequenceMatcher(None, " ".join(tw[i:i + k]), phrase).ratio() >= thr:
                return True
    return False

def classify_text(text):
    t = norm_text(text)
    for tid in MATCH_ORDER:
        for ph in TEMPLATES[tid]["match"]:
            if fuzzy_contains(t, norm_text(ph)):
                return tid, ph
    return None, None

def override_for(pdf_rel):
    for pat, tid in TEMPLATE_OVERRIDE.items():
        if fnmatch.fnmatch(pdf_rel.lower(), pat.lower()) or fnmatch.fnmatch(Path(pdf_rel).name.lower(), pat.lower()):
            return tid
    return None

def decide_template(pdf_rel, header, text_layer):
    """-> (template_id or None, mis_variant or None, evidence dict)"""
    title_text = " | ".join(map(str, (header or {}).get("title_lines") or []))
    detected, phrase = classify_text(title_text)
    source = "header" if detected else None
    if not detected and header:                                       # heading split oddly: try every field
        detected, phrase = classify_text(" ".join(str(v) for v in header.values() if v))
        source = "header_fields" if detected else None
    if not detected and text_layer:
        detected, phrase = classify_text(text_layer[:3000])
        source = "pdf_text_layer" if detected else None
    forced = override_for(pdf_rel)
    ev = dict(title_read=title_text or None, detected=detected, matched_phrase=phrase, detected_from=source,
              override=forced, strict_input=STRICT_INPUT, warnings=[])
    if STRICT_INPUT:
        chosen = forced
        if not forced:
            ev["warnings"].append("STRICT_INPUT and no TEMPLATE_OVERRIDE matches this file")
    else:
        chosen = forced or detected
    variant = None
    if chosen and ":" in chosen:
        chosen, variant = chosen.split(":", 1)
    if chosen and chosen not in TEMPLATES:
        ev["warnings"].append(f"unknown template id {chosen!r}"); chosen = None
    if variant and variant not in MIS_VARIANTS:
        ev["warnings"].append(f"unknown MIS sub-type {variant!r}"); variant = None
    if forced and detected and forced.split(":")[0] != detected:
        ev["warnings"].append(f"override {forced} differs from header ({detected})")
    ev["chosen"] = chosen
    return chosen, variant, ev

def detect_mis_variant(header, column_labels):
    title = norm_text(" ".join(map(str, (header or {}).get("title_lines") or [])))
    doc = norm_text((header or {}).get("doc_no"))
    labels = [norm_text(l) for l in column_labels if l]
    def has(markers):        # whole-label match (fuzzy), so "OK" does not match inside another word
        return any(l == norm_text(m) or l.startswith(norm_text(m) + " ")
                   or difflib.SequenceMatcher(None, l, norm_text(m)).ratio() >= 0.85
                   for l in labels for m in markers)
    if fuzzy_contains(title, "FI VISUAL INSPECTION"):
        return "FI_VISUAL_POWDER_COATING", "title FI VISUAL INSPECTION"
    if has(PLATING_MARKERS):
        return "PLATING_DAILY", "plating columns"
    if doc.endswith("/PL") or "/PL " in doc + " ":
        return "PLATING_DAILY", "doc no /PL"
    if labels:
        if has(DISPOSITION_MARKERS):
            return "POWDER_COATING_DAILY_QA", "powder-coating columns + disposition columns"
        return "POWDER_COATING_DAILY", "powder-coating columns"
    return None, "no columns read"

## 7. Common helpers: numbers, ditto, signatures

In [ ]:
DASH_RE = re.compile(r"^[\-—–~_=\s]+$")

def to_int(v):
    if v is None or isinstance(v, bool): return None
    if isinstance(v, int): return v
    if isinstance(v, float): return int(v) if v.is_integer() else None
    s = re.sub(r"[,\s]", "", str(v))
    return int(s) if re.fullmatch(r"-?\d+", s) else None

def to_num(v):
    """MIS / straight-pass cell: dash -> 0, empty -> None, '82%' -> 82, unreadable -> 'INVALID'."""
    if v is None or isinstance(v, bool): return None
    if isinstance(v, (int, float)): return int(v) if float(v).is_integer() else float(v)
    s = str(v).strip().replace(",", "").replace("%", "").strip()
    if s == "" or s.lower() in ("null", "none"): return None
    if DASH_RE.match(s) or s.lower() == "nil": return 0
    try:
        f = float(s); return int(f) if f.is_integer() else f
    except ValueError:
        return "INVALID"

def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s or "").lower()).strip("_") or "col"

SIGNATURE_PROMPT = """This is the bottom part of a scanned factory form with signature boxes.
For every printed signature label (e.g. "PREPARED BY", "QA HEAD", "HOD QA"), say whether the box has a signature or
handwritten name/initials.
Return only JSON: {"signatures": [{"label": printed label, "signed": true or false}]}"""

def read_signatures(ctx, img, y0, keys):
    H = img.shape[0]
    y0 = int(max(0, min(y0, H - 50)))
    crop = img[y0:min(H, y0 + int(0.2 * H))]
    if crop.shape[0] < 40:
        return {k: None for k in keys}
    hint = f'\nUse these keys if they fit the printed labels: {", ".join(keys)}.' if keys else ""
    parsed, _ = call_model(ctx, "signatures", with_margin(crop), SIGNATURE_PROMPT + hint, 512)
    out = {k: None for k in keys}
    for s in (parsed or {}).get("signatures", []) if isinstance(parsed, dict) else []:
        lab = slug(s.get("label"))
        key = next((k for k in keys if k == lab or difflib.SequenceMatcher(None, k, lab).ratio() >= 0.75), lab)
        out[key] = bool(s.get("signed"))
    return out

## 8. Record forms (supplier reports, line rejection, scrap note)
Same method as `Qwen3VL_Form_Extraction.ipynb`: find the handwritten rows, group wrapped lines into one entry and send
`[column-header strip + entry]` per entry.

In [ ]:
def record_prompt(tpl):
    lines = [
        f'This image is cut from a handwritten factory QA register: "{tpl["title"]}" (Meenakshi Polymers, Haridwar).',
        "The top strip shows the printed column headers. Below it is ONE handwritten entry. "
        "The entry may continue onto a second ruled line - treat those lines as the same entry.",
        "Extract the entry into JSON with exactly these keys:"]
    for key, (header, typ) in tpl["fields"].items():
        lines.append(f'- "{key}": {typ} or null   (printed column: "{header}")')
    lines += [
        "Rules:",
        "- Copy exactly what is handwritten. Do NOT calculate, correct, or infer any value.",
        "- Empty or illegible cell -> null. A struck-out value is ignored; use the corrected value.",
        '- Integer fields: digits only, e.g. "07" -> 7, "12,274" -> 12274.',
        '- In text fields, a ditto mark meaning "same as above" (", 〃, or two short strokes like 11) -> "DITTO".',
        "- Keep spelling as written. Join text wrapped onto the next line with a single space.",
        "- Dates: copy as written, e.g. 03/6/26.",
        'Return only JSON in this form: {"entries": [ { ...keys above... } ]}']
    return "\n".join(lines)

DATE_RE = re.compile(r"^\s*\d{1,2}\s*[/.\-]\s*\d{1,2}\s*[/.\-]\s*\d{2,4}\s*$")

def extract_record(ctx, img, raw_gray, tpl):
    dbg, warnings = ctx.dbg, []
    gray, bw, h, v, rows, cands = basic_grid(img)
    dbg.save("binary", bw)
    if len(rows) < tpl["header_rows"] + tpl["footer_rows"] + 2 or len(cands) < 2:
        raise RuntimeError(f"table grid not found (rows={len(rows)}, column lines={len(cands)})")
    cols, col_report = filter_columns(raw_gray, rows, cands)
    if len(cols) < 2:
        raise RuntimeError("no column lines left after filtering")
    if len(cols) != tpl["expected_col_lines"]:
        warnings.append(f"found {len(cols)} column lines, template expects {tpl['expected_col_lines']}")
    ink = ink_mask(gray, h, v);                   dbg.save("ink_no_lines", ink)
    x0, x1, hr = cols[0], cols[-1], tpl["header_rows"]
    data_rows = range(hr, len(rows) - 1 - tpl["footer_rows"])
    row_ink = {i: int(ink[rows[i] + 4:rows[i + 1] - 4, x0:x1].sum() / 255) for i in data_rows}
    filled = [i for i, s in row_ink.items() if s > MIN_ROW_INK]

    def first_col_ink(i):
        return ink[rows[i] + 4:rows[i + 1] - 4, cols[0] + 4:cols[1] - 4].sum() / 255
    records = []
    for i in filled:
        if records and i == records[-1][-1] + 1 and first_col_ink(i) < CONTINUATION_MAX_INK:
            records[-1].append(i)
        else:
            records.append([i])
    if not records:
        warnings.append("no handwritten rows found")

    ov = img.copy()       # rows red, kept columns blue, dropped columns yellow, header magenta, records green
    for i, y in enumerate(rows):
        cv2.line(ov, (0, y), (ov.shape[1], y), (0, 0, 255), 2)
        cv2.putText(ov, f"r{i}", (5, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    for c in col_report:
        cv2.line(ov, (c["x"], 0), (c["x"], ov.shape[0]), (255, 0, 0) if c["kept"] else (0, 200, 255), 2 if c["kept"] else 5)
    cv2.rectangle(ov, (x0, rows[hr - 1]), (x1, rows[hr]), (255, 0, 255), 5)
    for k, rec in enumerate(records):
        cv2.rectangle(ov, (x0, rows[rec[0]]), (x1, rows[rec[-1] + 1]), (0, 170, 0), 6)
        cv2.putText(ov, f"entry {k}", (x0 + 10, rows[rec[0]] + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 170, 0), 3)
    ctx.overlay = dbg.save("grid_overlay", ov)
    if SHOW_INLINE:
        show(ov[:min(ov.shape[0], rows[-1] + 100)], f"{ctx.pdf_rel} p{ctx.page_no} – rows red · columns blue "
             "(yellow = dropped shadow line) · header magenta · entries green", width=12)

    xa, xb = max(x0 - SIDE_PAD, 0), min(x1 + SIDE_PAD, img.shape[1])
    header = img[rows[hr - 1]:rows[hr], xa:xb]
    prompt, entries_raw = record_prompt(tpl), []
    for k, rec in enumerate(records):
        top_pad = 0 if (rec[0] == hr or (rec[0] - 1) in filled) else TOP_PAD
        body = img[rows[rec[0]] - top_pad:min(rows[rec[-1] + 1] + BOTTOM_PAD, img.shape[0]), xa:xb]
        parsed, call = call_model(ctx, f"entry_{k:02d}", with_margin(np.vstack([header, body])), prompt, 1024)
        entries_raw.append((k, rec, parsed, call))

    entries, issues, prev = [], [], {}
    for k, rec, parsed, call in entries_raw:
        got = (parsed or {}).get("entries") if isinstance(parsed, dict) else None
        tag = f"entry {k} (rows {rec})"
        if call["error"] and not DRY_RUN:
            issues.append(f"{tag}: {call['error']}")
        if got is not None and len(got) != 1:
            issues.append(f"{tag}: model returned {len(got)} entries for one crop")
        e = got[0] if got else {}
        out = {}
        for key, (_, typ) in tpl["fields"].items():
            val = e.get(key) if isinstance(e, dict) else None
            if typ == "integer":
                iv = to_int(val)
                if val not in (None, "") and iv is None:
                    issues.append(f"{tag}: {key} not a number ({val!r})")
                out[key] = iv
            else:
                s = None if val is None else str(val).strip() or None
                if s is not None and s.upper() == "DITTO":
                    s = prev.get(key)
                out[key] = s
        if not DRY_RUN:
            for key in tpl["required"]:
                if out.get(key) is None:
                    issues.append(f"{tag}: missing {key}")
            for desc, needed, fn in tpl["rules"]:
                if any(out.get(x) is None for x in needed):
                    continue
                try:
                    if not fn(out):
                        issues.append(f"{tag}: rule failed: {desc}")
                except Exception as ex:
                    issues.append(f"{tag}: rule error: {desc} ({ex})")
            if out.get("date") and not DATE_RE.match(out["date"]):
                issues.append(f"{tag}: date format {out['date']!r}")
        prev = {**prev, **{x: y for x, y in out.items() if y is not None}}
        entries.append(out)

    sig_y = rows[len(rows) - 1 - tpl["footer_rows"]] if len(rows) > tpl["footer_rows"] + 1 else rows[-1]
    signatures = read_signatures(ctx, img, sig_y, tpl["signature_keys"])
    return dict(entries=entries, signatures=signatures), issues, warnings

## 9. MIS defect matrix (one template, sub-types)
1. **Locate** the column-header block and the table on the page (Qwen grounding on a half-size page; fallback: ruled lines).
2. **Read the columns** from the header strip as leaf columns + group heading (two-level headers).
3. **Sub-type** from title / doc no / columns -> canonical keys (aligned to the read labels).
4. **Bands**: the table is cut into overlapping bands of ~6 rows; each band image = `[header strip + band]`;
   the model returns every printed row (SR.NO, defect label, reason, all columns).
5. **Assemble** rows into sections (a TOTAL row closes a section, a %/straight-pass row follows it) and summary rows.
6. **Validate** row sums, disposition sums and section totals.

In [ ]:
MIS_LOCATE_PROMPT = """This is a scanned daily MIS defect sheet with a large table.
Locate:
1. "header": the table's printed column-header block - the row(s) holding the column names such as SR.NO, DEFECT,
   REASON, part names and TOTAL, including a group heading above some columns (e.g. "FOR REWORK"). Not the company
   title, not the DATE / MONTH line.
2. "table": the whole table from the column-header block down to its last printed row (summary rows such as
   TOTAL DEFECT QTY or PRODUCTION included), not the signatures below it.
Return only JSON: {"header": [x1, y1, x2, y2], "table": [x1, y1, x2, y2]} with coordinates on a 0-1000 scale
relative to the image width and height."""

MIS_COLUMNS_PROMPT = """This is the printed column-header block of a factory defect table.
Some columns have a two-level heading: a group heading on top spans several columns (for example "FOR REWORK" above
"MINOR" and "MAJOR", or "REJECTION" above "SCRAP"), while other columns (for example "TOTAL") span both levels.
List the bottom-level columns from left to right - one entry for every column in which values are written,
including SR.NO / DEFECT / REASON. For each give the printed label (join the words of a label printed on several
lines with a space) and the group heading printed above it, or null when there is none.
Return only JSON: {"columns": [{"label": "SR.NO", "group": null}, {"label": "MINOR", "group": "FOR REWORK"}]}"""

def box_px(b, W, H, view_wh):
    """[x1,y1,x2,y2] from the model -> page pixels. Qwen3-VL answers on a 0-1000 scale; absolute model-view pixels
    are accepted too (any value > 1000)."""
    try:
        b = [float(x) for x in b][:4]
    except (TypeError, ValueError):
        return None
    if len(b) != 4:
        return None
    if max(b) > 1000 and view_wh:
        sx, sy = W / view_wh[0], H / view_wh[1]
    else:
        sx, sy = W / 1000, H / 1000
    x1, y1, x2, y2 = b[0] * sx, b[1] * sy, b[2] * sx, b[3] * sy
    return [int(min(x1, x2)), int(min(y1, y2)), int(max(x1, x2)), int(max(y1, y2))]

def mis_layout(ctx, img):
    H, W = img.shape[:2]
    layout, notes = None, []
    if not DRY_RUN:
        small = cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
        parsed, call = call_model(ctx, "mis_locate", small, MIS_LOCATE_PROMPT, 200)
        if isinstance(parsed, dict):
            # the box was given for the half-size image: the scale factors are relative, so map with W, H directly;
            # absolute answers are relative to the processed view of the half-size image
            hb = box_px(parsed.get("header"), W, H, call.get("model_view_wh"))
            tb = box_px(parsed.get("table"), W, H, call.get("model_view_wh"))
            if hb and tb:
                hh = hb[3] - hb[1]
                ok = (0.008 * H <= hh <= 0.15 * H and tb[3] > hb[3] + 0.2 * H and tb[2] - tb[0] > 0.5 * W
                      and tb[1] <= hb[3])
                if ok:
                    layout = dict(header_y=(max(hb[1] - 6, 0), hb[3] + 4), x=(max(tb[0] - 25, 0), min(tb[2] + 25, W)),
                                  bottom=min(tb[3] + 15, H), method="model")
                else:
                    notes.append(f"model boxes rejected by sanity check: header {hb}, table {tb}")
            else:
                notes.append(f"model boxes unreadable: {parsed}")
    lines = find_line_pieces(horizontal_lines(binarize(to_gray(img))))
    if layout is None:
        # fallback: column lines that reach the table bottom start at the header top; the next long row line ends it
        gray, bw, h, v, rows, cands = basic_grid(img)
        vm = vertical_lines(bw)
        tops = []
        for x in cands:
            col = vm[:, max(x - 3, 0):x + 4].max(axis=1)
            ys = np.where(col > 0)[0]
            if len(ys) > 0.3 * H:
                tops.append(int(ys.min()))
        long_lines = [q for q in lines if q["x1"] - q["x0"] > 0.45 * W]
        if len(tops) >= 3 and long_lines:
            y_top = int(np.median(tops))
            below = [q["y"] for q in long_lines if q["y"] > y_top + 28]
            if below:
                layout = dict(header_y=(y_top, below[0]), x=(max(min(cands) - 25, 0), min(max(cands) + 25, W)),
                              bottom=max(q["y"] for q in long_lines) + 15, method="ruled_lines")
    if layout is None:
        layout = dict(header_y=(int(0.06 * H), int(0.12 * H)), x=(int(0.03 * W), int(0.97 * W)), bottom=int(0.9 * H),
                      method="fixed_fraction")
        notes.append("table not located - fixed fallback used, check the overlay")
    layout["row_lines"] = [q["y"] for q in lines if q["x1"] - q["x0"] > 0.3 * (layout["x"][1] - layout["x"][0])]
    layout["notes"] = notes
    return layout

def mis_bands(layout):
    y, end, bands = layout["header_y"][1], layout["bottom"], []
    lines = layout["row_lines"]
    def snap(t):
        near = [l for l in lines if abs(l - t) <= MIS_SNAP_PX]
        return min(near, key=lambda l: abs(l - t)) if near else t
    while y < end - 20:
        b = min(snap(y + MIS_BAND_PX), end)
        if end - b < MIS_BAND_PX * 0.35:                  # do not leave a sliver at the bottom
            b = end
        bands.append((max(y - (MIS_BAND_OVERLAP if bands else 0), layout["header_y"][1]), b + (BOTTOM_PAD if b < end else 0)))
        y = b
    return bands

def align_columns(read, canon):
    """Needleman-Wunsch style alignment of read labels to canonical labels -> list of (read_idx, canon_idx|None)."""
    def text(c): return norm_text(f"{c.get('group') or ''} {c.get('label') or ''}")
    n, m = len(read), len(canon)
    sim = [[difflib.SequenceMatcher(None, text(read[i]), text(canon[j])).ratio() for j in range(m)] for i in range(n)]
    S = np.zeros((n + 1, m + 1)); B = np.zeros((n + 1, m + 1), dtype=int)
    for i in range(1, n + 1): S[i][0], B[i][0] = S[i - 1][0] - 0.3, 1
    for j in range(1, m + 1): S[0][j], B[0][j] = S[0][j - 1] - 0.3, 2
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            opts = [S[i - 1][j - 1] + (sim[i - 1][j - 1] if sim[i - 1][j - 1] >= 0.45 else -1), S[i - 1][j] - 0.3, S[i][j - 1] - 0.3]
            B[i][j] = int(np.argmax(opts)); S[i][j] = max(opts)
    pairs, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and B[i][j] == 0:
            pairs.append((i - 1, j - 1, sim[i - 1][j - 1])); i -= 1; j -= 1
        elif i > 0 and (j == 0 or B[i][j] == 1):
            pairs.append((i - 1, None, 0)); i -= 1
        else:
            pairs.append((None, j - 1, 0)); j -= 1
    return pairs[::-1]

LABEL_KEYS = {"sr_no": ["SR NO", "S NO", "SR N", "SNO", "SR"], "defect": ["DEFECT", "DESCRIPTION OF DEFECT", "DESCRIPTION"],
              "reason": ["REASON"]}

def assign_roles(cols):
    seen_total = False
    for c in cols:
        if c["key"] in LABEL_KEYS:
            c["role"] = "label"
        elif c["key"] == "grand_total" or (seen_total and norm_text(c["label"]) == "TOTAL"):
            c["role"], c["key"] = "grand_total", "grand_total"
        elif c["key"] == "total" or norm_text(c["label"]) == "TOTAL":
            c["role"], c["key"], seen_total = "total", "total", True
        else:
            c["role"] = "disposition" if seen_total else "part"
    keys = set()
    for c in cols:                                     # unique keys
        k, n = c["key"], 2
        while c["key"] in keys:
            c["key"] = f"{k}_{n}"; n += 1
        keys.add(c["key"])
    return cols

def mis_columns(read, variant):
    """read: [{label, group}] from the page. Returns (columns, warnings)."""
    warnings = []
    read = [dict(label=str(c.get("label") or "").strip(), group=(str(c["group"]).strip() if c.get("group") else None))
            for c in (read or []) if isinstance(c, dict) and str(c.get("label") or "").strip()]
    canon = [dict(c) for c in MIS_VARIANTS[variant]["columns"]] if variant else None
    if not read:
        if canon:
            warnings.append("columns not read from the page - using the sub-type's column list")
            return assign_roles(canon), warnings
        return [], ["columns not read and no sub-type known"]
    cols = []
    if canon:
        for ri, ci, s in align_columns(read, canon):
            if ri is None:
                warnings.append(f"sub-type column {canon[ci]['label']!r} not found on the page")
                continue
            r = read[ri]
            if ci is None:
                key = slug(f"{r['group']} {r['label']}" if r["group"] else r["label"])
                warnings.append(f"column {r['label']!r} (group {r['group']}) is not in sub-type {variant}")
                cols.append(dict(label=r["label"], key=key, group=r["group"]))
            else:
                cols.append(dict(label=r["label"], key=canon[ci]["key"], group=r["group"] or canon[ci]["group"]))
    else:
        for r in read:
            n = norm_text(r["label"])
            key = next((k for k, alts in LABEL_KEYS.items() if n in alts or any(n.startswith(a) for a in alts[:2])), None)
            cols.append(dict(label=r["label"], group=r["group"],
                             key=key or slug(f"{r['group']} {r['label']}" if r["group"] else r["label"])))
    return assign_roles(cols), warnings

def mis_band_prompt(cols, variant):
    value_cols = [c for c in cols if c["role"] != "label"]
    has_reason = any(c["key"] == "reason" for c in cols)
    title = MIS_VARIANTS[variant]["title"] if variant else "daily MIS defect sheet"
    lines = [
        f'This image is cut from a handwritten factory sheet: "{title}" (Meenakshi Polymers).',
        "The top strip is the printed column header. Some columns sit under a group heading (for example MINOR and "
        "MAJOR under FOR REWORK). Below it is a horizontal band of the table with several printed rows.",
        "For EVERY printed row in the band, top to bottom, return one object with these keys:",
        '- "sr_no": printed serial number (integer) or null',
        '- "label": the printed row text in the DEFECT / DESCRIPTION column, e.g. "PEEL OFF", '
        '"TOTAL PLATING DEFECT", "PRODUCTION"']
    if has_reason:
        lines.append('- "reason": printed text in the REASON column or null')
    for c in value_cols:
        g = f' under "{c["group"]}"' if c.get("group") else ""
        lines.append(f'- "{c["key"]}": number or null   (column "{c["label"]}"{g})')
    lines += [
        "Rules:",
        "- Columns are listed left to right in the same order as on the sheet. Count the columns carefully.",
        "- Copy exactly what is written. Do NOT calculate, total or correct anything.",
        "- A dash (-, —, ~) written in a cell = 0. An empty cell = null. Percent values: the number without %.",
        "- A struck-out value is ignored; use the corrected value.",
        "- A number written on the line between two rows belongs to the row it is closest to.",
        "- Skip a row cut off at the top or bottom edge only if its printed label is not readable.",
        "- A heading row without cells (e.g. FABRICATION DEFECT) is returned with all values null.",
        'Return only JSON: {"rows": [ {...}, ... ]}']
    return "\n".join(lines)

def row_key(r):
    return (r.get("sr_no"), norm_text(r.get("label")))

def same_row(a, b):
    if a.get("sr_no") is not None and b.get("sr_no") is not None and a["sr_no"] != b["sr_no"]:
        return False
    la, lb = norm_text(a.get("label")), norm_text(b.get("label"))
    if not la or not lb:
        return a.get("sr_no") is not None and a.get("sr_no") == b.get("sr_no")
    return difflib.SequenceMatcher(None, la, lb).ratio() >= 0.8

def merge_bands(band_rows, value_keys):
    """Rows of overlapping bands -> one list; a row seen in two neighbouring bands is kept once
    (the copy with more filled cells - the other one is usually cut by the band edge)."""
    out = []
    for b, rows in enumerate(band_rows):
        for r in rows:
            filled = sum(r.get(k) is not None for k in value_keys)
            dup = next((o for o in out[-5:] if o["_band"] == b - 1 and same_row(o, r)), None)
            if dup is not None:
                if filled > dup["_filled"]:
                    dup.update(r, _band=dup["_band"], _filled=filled, _dup=True)
                else:
                    dup["_dup"] = True
                continue
            out.append(dict(r, _band=b, _filled=filled))
    return out

TOTAL_RE   = re.compile(r"\bTOTAL\b")
PCT_RE     = re.compile(r"%|STRAIGHT ?PASS|PPM")
SUMMARY_RE = re.compile(r"OVER ?ALL|PRODUCTION|REWORK PLANT|CUSTOMER RETURN|TOTAL DEFECT QTY|TOTAL REJECTION|REVISION")

def section_name(total_label, heading, first_row):
    if heading:
        return heading
    name = re.sub(r"\b(TOTAL|DEFECTS?|QTY|REJECTION|OF THE DAY)\b", " ", norm_text(total_label or "")).strip()
    if name:
        return re.sub(r"\s+", " ", name)
    return (first_row or {}).get("reason") or None

def assemble_mis(rows, cols):
    keys = [c["key"] for c in cols if c["role"] != "label"]
    has_reason = any(c["key"] == "reason" for c in cols)
    def shape(r, data):
        o = {"sr_no": to_int(r.get("sr_no")) if data else None, "defect": r.get("label")}
        if has_reason:
            o["reason"] = r.get("reason") if data else None
        o.update({k: to_num(r.get(k)) for k in keys})
        return o
    sections, summary, cur, heading, in_summary = [], [], None, None, False
    for r in rows:
        label = norm_text(r.get("label"))
        sr = to_int(r.get("sr_no"))
        has_vals = any(r.get(k) is not None for k in keys)
        is_total, is_pct = bool(TOTAL_RE.search(label)) and sr is None, bool(PCT_RE.search(label)) and sr is None
        if sr is not None and not in_summary:
            if cur is None or cur["section_total"] is not None:
                cur = dict(section=None, rows=[], section_total=None, section_pct=None, _heading=heading)
                sections.append(cur); heading = None
            cur["rows"].append(shape(r, True))
        elif is_pct and cur is not None and cur["section_total"] is not None and cur["section_pct"] is None and not in_summary:
            cur["section_pct"] = shape(r, False)
        elif (is_total and not is_pct and not SUMMARY_RE.search(label) and cur is not None and cur["section_total"] is None
              and cur["rows"] and not in_summary):
            cur["section_total"] = shape(r, False)
        elif not has_vals and not is_total and not is_pct and label and not SUMMARY_RE.search(label) and not in_summary:
            heading = r.get("label")                         # e.g. "FABRICATION DEFECT"
        elif label or has_vals:
            in_summary = in_summary or bool(cur and cur["rows"])
            summary.append(shape(r, False))
    for s in sections:
        s["section"] = section_name((s["section_total"] or {}).get("defect"), s.pop("_heading"), (s["rows"] or [None])[0])
    return sections, summary

def validate_mis(sections, summary, cols):
    issues = []
    parts = [c["key"] for c in cols if c["role"] == "part"]
    disp = [c["key"] for c in cols if c["role"] == "disposition"]
    has_total = any(c["role"] == "total" for c in cols)
    grand = "grand_total" if any(c["role"] == "grand_total" for c in cols) else ("total" if has_total else None)
    def num(x): return x if isinstance(x, (int, float)) and not isinstance(x, bool) else 0
    def check_row(r, tag):
        bad = [k for k, v in r.items() if v == "INVALID"]
        if bad:
            issues.append(f"{tag}: unreadable number in {bad}")
        if has_total and r.get("total") is not None and any(r.get(k) is not None for k in parts):
            s = sum(num(r.get(k)) for k in parts)
            if s != num(r["total"]):
                issues.append(f"{tag}: parts sum {s} ≠ TOTAL {r['total']}")
        if disp and grand and r.get(grand) is not None and any(r.get(k) is not None for k in disp):
            s = sum(num(r.get(k)) for k in disp)
            if s != num(r[grand]):
                issues.append(f"{tag}: disposition sum {s} ≠ {grand} {r[grand]}")
    for s in sections:
        for r in s["rows"]:
            check_row(r, f"row {r.get('sr_no')} {r.get('defect')}")
        t = s["section_total"]
        if t:
            check_row(t, f"{s['section']} total")
            for k in parts + disp + (["total"] if has_total else []):
                if t.get(k) is not None and any(r.get(k) is not None for r in s["rows"]):
                    sm = sum(num(r.get(k)) for r in s["rows"])
                    if sm != num(t[k]):
                        issues.append(f"{s['section']} total, column {k}: rows sum {sm} ≠ {t[k]}")
    srs = [r["sr_no"] for s in sections for r in s["rows"] if r.get("sr_no") is not None]
    gaps = sorted(set(range(min(srs), max(srs) + 1)) - set(srs)) if srs else []
    if gaps:
        issues.append(f"serial numbers missing (row not read?): {gaps[:15]}")
    return issues

def extract_mis(ctx, img, header, variant):
    warnings = []
    layout = mis_layout(ctx, img)
    warnings += layout["notes"]
    (hy0, hy1), (xa, xb), bottom = layout["header_y"], layout["x"], layout["bottom"]
    header_strip = img[hy0:hy1, xa:xb]
    bands = mis_bands(layout)
    ov = img.copy()
    cv2.rectangle(ov, (xa, hy0), (xb, hy1), (255, 0, 255), 6)
    cv2.rectangle(ov, (xa, hy0), (xb, bottom), (255, 0, 0), 3)
    for y in layout["row_lines"]:
        cv2.line(ov, (xa, y), (xb, y), (0, 0, 255), 1)
    for k, (b0, b1) in enumerate(bands):
        c = (0, 170, 0) if k % 2 == 0 else (0, 120, 255)
        cv2.rectangle(ov, (xa + 8 + 8 * (k % 2), b0), (xb - 8 - 8 * (k % 2), b1), c, 4)
        cv2.putText(ov, f"band {k}", (xb - 230, b0 + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, c, 3)
    ctx.overlay = ctx.dbg.save("mis_layout", ov)
    if SHOW_INLINE:
        show(ov, f"{ctx.pdf_rel} p{ctx.page_no} – MIS layout ({layout['method']}): header magenta · table blue · "
             "bands green/orange", width=10)

    parsed, _ = call_model(ctx, "mis_columns", with_margin(header_strip), MIS_COLUMNS_PROMPT, 1024)
    read = (parsed or {}).get("columns") if isinstance(parsed, dict) else None
    labels = [c.get("label") for c in read or [] if isinstance(c, dict)]
    how = "override"
    if variant is None:
        variant, how = detect_mis_variant(header, labels)
    cols, w = mis_columns(read, variant)
    warnings += w
    if not cols:
        raise RuntimeError("MIS columns unknown (no column header read, no sub-type)")
    prompt = mis_band_prompt(cols, variant)
    value_keys = [c["key"] for c in cols if c["role"] != "label"]
    band_rows = []
    for k, (b0, b1) in enumerate(bands):
        crop = with_margin(np.vstack([header_strip, img[b0:b1, xa:xb]]))
        parsed, _ = call_model(ctx, f"band_{k:02d}", crop, prompt, MAX_NEW_TOKENS)
        rows = (parsed or {}).get("rows") if isinstance(parsed, dict) else None
        band_rows.append([r for r in rows or [] if isinstance(r, dict)])
    merged = merge_bands(band_rows, value_keys)
    sections, summary = assemble_mis(merged, cols)
    issues = [] if DRY_RUN else validate_mis(sections, summary, cols)
    if not DRY_RUN and not merged:
        issues.append("no rows read")
    signatures = read_signatures(ctx, img, bottom - 10, TEMPLATES["MIS"]["signature_keys"])
    groups = {}
    for c in cols:
        if c.get("group"):
            groups.setdefault(slug(c["group"]), []).append(c["key"])
    data = dict(mis_variant=variant, mis_variant_reason=how,
                columns=[c["key"] for c in cols if c["role"] != "label"],
                column_meta=[dict(key=c["key"], label=c["label"], group=c.get("group"), role=c["role"]) for c in cols],
                column_groups=groups, sections=sections, summary=summary, signatures=signatures)
    ctx.extra = dict(layout={k: v for k, v in layout.items() if k != "row_lines"}, bands=bands,
                     rows_seen_twice=sum(1 for r in merged if r.get("_dup")))
    return data, issues, warnings

## 10. Straight-pass monitoring sheet
Landscape grid: parts (8 parameter rows each) × days (A / B / TOTAL columns). One model call per part × day that
has handwriting; the crop is `[part labels + parameter names | that day's three columns]` with the day header on top.

In [ ]:
DAYS_PROMPT = """This strip is the printed header row of a monitoring sheet with day columns.
List the day numbers printed in the top row, left to right.
Return only JSON: {"days": ["21", "22", ...]}"""

def st_prompt(tpl):
    params = "\n".join(f'   "{k}": number or null   (row "{lab}")' for k, lab in tpl["params"])
    return f"""This crop is from the handwritten "{tpl['title']}". Left: printed part details (S.NO, PART NAME, PART NO,
MODEL) and the 8 parameter rows. Right: ONE day with three columns A, B and TOTAL (the day number is printed on top).
Return only JSON:
{{"sno": integer or null, "part_name": string or null, "part_no": string or null, "model": string or null,
 "day": printed day number as string or null,
 "whole_day_no_plan": true if "NO PLAN" is written across all three columns of this day, else false,
 "A": SHIFT, "B": SHIFT, "TOTAL": SHIFT}}
where SHIFT is null when that column is empty, "NO PLAN" when NO PLAN is written down that column only, or
{{
{params}
}}
Rules: copy exactly what is written, do not calculate. Percent values: number without %. Empty cell -> null.
A struck-out value is ignored."""

def snap_to(t, lines, tol):
    near = [l for l in lines if abs(l - t) <= tol]
    return int(min(near, key=lambda l: abs(l - t))) if near else int(round(t))

def extract_stpass(ctx, img, raw_gray, tpl):
    """The sheet is printed on a uniform grid, so part blocks (n × 8 rows) and day groups (3 columns each) are found by
    splitting the data area evenly and snapping to the nearest ruled line - a faint or missing line does not matter."""
    warnings, issues = [], []
    W = img.shape[1]
    gray, bw, h, v, rows, cands = basic_grid(img)
    cols, col_report = filter_columns(raw_gray, rows, cands)
    pitches = list(np.diff(rows))
    run = uniform_run(pitches, tol=0.2, min_len=8)
    if run is None or len(cols) < 4:
        raise RuntimeError(f"straight-pass grid not found (rows={len(rows)}, cols={len(cols)})")
    s = run[0]
    med = float(np.median(pitches[run[0]:run[1] + 1]))
    j = s
    while j + 1 < len(rows) and rows[j + 1] - rows[j] <= 1.6 * med:      # table bottom
        j += 1
    top_y, bottom_y = rows[s], rows[j]
    n_par = len(tpl["params"])
    n_parts = max(1, int(round((bottom_y - top_y) / (n_par * med))))
    step = (bottom_y - top_y) / n_parts
    bounds = [snap_to(top_y + k * step, rows, 0.4 * med) for k in range(n_parts + 1)]
    blocks = list(zip(bounds[:-1], bounds[1:]))
    # label area = columns up to the widest column in the left part (PARAMETER); day area = the rest
    left = [(cols[i + 1] - cols[i], i) for i in range(len(cols) - 1) if cols[i + 1] < 0.45 * W]
    label_edge = cols[max(left)[1] + 1] if left else int(0.3 * W)
    right = cols[-1]
    hdr_top = rows[max(s - tpl["header_bands"], 0)]
    ink = ink_mask(gray, h, v)

    parsed, _ = call_model(ctx, "days", with_margin(img[hdr_top:top_y, label_edge:right]), DAYS_PROMPT, 256)
    days_read = [str(d) for d in ((parsed or {}).get("days") or [])] if isinstance(parsed, dict) else []
    n_days = len(days_read) if 4 <= len(days_read) <= 16 else tpl["days_per_sheet"]
    if days_read and len(days_read) != n_days:
        warnings.append(f"read {len(days_read)} day numbers - using {n_days} day groups")
    day_w = (right - label_edge) / n_days
    gx = [snap_to(label_edge + g * day_w, cols, 0.3 * day_w / 3) for g in range(n_days + 1)]
    groups = list(zip(gx[:-1], gx[1:]))

    ov = img.copy()
    for y in rows:
        cv2.line(ov, (0, y), (W, y), (0, 0, 255), 1)
    for x in cols:
        cv2.line(ov, (x, 0), (x, ov.shape[0]), (255, 0, 0), 1)
    cv2.rectangle(ov, (cols[0], hdr_top), (label_edge, top_y), (255, 0, 255), 5)
    for k, (y0, y1) in enumerate(blocks):
        cv2.rectangle(ov, (cols[0] + 4, y0 + 4), (right - 4, y1 - 4), (0, 170, 0), 4)
        cv2.putText(ov, f"part {k + 1}", (cols[0] + 10, y0 + 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 170, 0), 3)
    for g, (x0, x1) in enumerate(groups):
        cv2.rectangle(ov, (x0 + 3, hdr_top), (x1 - 3, bottom_y), (0, 120, 255), 3)
        cv2.putText(ov, f"d{g}", (x0 + 8, hdr_top - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 120, 255), 3)
    ctx.overlay = ctx.dbg.save("stpass_grid", ov)
    if SHOW_INLINE:
        show(ov, f"{ctx.pdf_rel} p{ctx.page_no} – straight-pass grid: label header magenta · {len(blocks)} parts green · "
             f"{len(groups)} day groups orange", width=14)

    prompt = st_prompt(tpl)
    keys = [k for k, _ in tpl["params"]]
    parts = []
    for bi, (y0, y1) in enumerate(blocks):
        part = dict(sno=None, part_name=None, part_no=None, model=None, days={})
        for g, (gx0, gx1) in enumerate(groups):
            amount = int(ink[y0 + 4:y1 - 4, gx0 + 4:gx1 - 4].sum() / 255)
            if amount < ST_MIN_DAY_INK:
                continue
            yb = min(y1 + BOTTOM_PAD, img.shape[0])
            label = np.vstack([img[hdr_top:top_y, cols[0]:label_edge], img[y0:yb, cols[0]:label_edge]])
            day = np.vstack([img[hdr_top:top_y, gx0:gx1], img[y0:yb, gx0:gx1]])
            sep = np.full((label.shape[0], 12, 3), 255, np.uint8)
            crop = with_margin(np.hstack([label, sep, day]))
            parsed, call = call_model(ctx, f"part{bi + 1}_day{g}", crop, prompt, 1024)
            d = parsed if isinstance(parsed, dict) else {}
            for k in ("sno", "part_name", "part_no", "model"):
                if part[k] is None and d.get(k) not in (None, ""):
                    part[k] = to_int(d[k]) if k == "sno" else str(d[k]).strip()
            day_no = str(d.get("day") or (days_read[g] if g < len(days_read) else g + 1))
            if d.get("whole_day_no_plan"):
                part["days"][day_no] = "NO PLAN"
                continue
            entry = {}
            for sh in tpl["shifts"]:
                val = d.get(sh)
                if isinstance(val, str) and "PLAN" in val.upper():
                    entry[sh] = "NO PLAN"
                elif isinstance(val, dict):
                    vals = {k: to_num(val.get(k)) for k in keys}
                    entry[sh] = vals if any(x is not None for x in vals.values()) else None
                else:
                    entry[sh] = None
            if any(x is not None for x in entry.values()) or DRY_RUN:
                part["days"][day_no] = entry
            if call["error"] and not DRY_RUN:
                issues.append(f"part {bi + 1} day {day_no}: {call['error']}")
        parts.append(part)

    if not DRY_RUN:                                         # plausibility checks
        for p in parts:
            for day, entry in p["days"].items():
                if not isinstance(entry, dict):
                    continue
                for sh, x in entry.items():
                    if not isinstance(x, dict):
                        continue
                    tag = f"part {p['sno']} day {day} {sh}"
                    if any(val == "INVALID" for val in x.values()):
                        issues.append(f"{tag}: unreadable number")
                        continue
                    c, o, rj, rw = x.get("checked_qty"), x.get("ok_qty"), x.get("rej_qty"), x.get("rew_qty")
                    if c is not None and o is not None and rw is not None and o + rw + (rj or 0) != c:
                        issues.append(f"{tag}: ok {o} + rew {rw} + rej {rj or 0} ≠ checked {c}")
                    pct = x.get("visual_defects_straight_pass_pct")
                    if pct is not None and c and o is not None and abs(pct - 100 * o / c) > 2:
                        issues.append(f"{tag}: straight pass {pct}% vs ok/checked {100 * o / c:.0f}%")
    days_printed = [to_int(d) if to_int(d) is not None else d for d in days_read] or None
    return dict(parts=parts, signatures={}, days_printed=days_printed), issues, warnings

## 11. Page and PDF driver
`ocr_pdf(path)` is the single entry point: it returns one expected-JSON object per page and writes
`json/<pdf>.json`. Run it on any PDF (also outside the batch below).

In [ ]:
def build_header(tpl_id, header, matched_phrase):
    keys = TEMPLATES[tpl_id]["header_keys"]
    h = header if isinstance(header, dict) else {}
    out = {k: h.get(k) for k in keys}
    if "title" in keys:
        lines = [str(t) for t in h.get("title_lines") or []]
        best = None
        if matched_phrase and lines:
            best = max(lines, key=lambda l: difflib.SequenceMatcher(None, norm_text(l), norm_text(matched_phrase)).ratio())
        out["title"] = best or (lines[0] if lines else None)
    if out.get("company") is None and "company" in keys:
        out["company"] = next((str(t) for t in h.get("title_lines") or [] if "MEENAKSHI" in str(t).upper()), None)
    return out

def process_page(pdf_path, pdf_rel, page, page_no, page_dir):
    ctx = PageCtx(pdf_rel, page_no, page_dir)
    ctx.overlay, ctx.extra = None, {}
    t0 = time.time()
    meta = dict(source_file=pdf_rel, page=page_no, debug_dir=str(page_dir))
    try:
        text_layer = page.get_text() or ""
        img, raw_gray, angle, info = prepare_page(page, ctx.dbg)
        meta.update(deskew_deg=round(angle, 2), render=info, size_wh=(img.shape[1], img.shape[0]),
                    text_layer_chars=len(text_layer.strip()))
        orient = "portrait" if img.shape[0] >= img.shape[1] else "landscape"
        header_img = img[:int(img.shape[0] * HEADER_BAND_FRAC[orient])]
        header, _ = call_model(ctx, "header", with_margin(header_img), HEADER_PROMPT, 768)
        tpl_id, variant, ev = decide_template(pdf_rel, header, text_layer)
        meta["template_decision"] = ev
        if VERBOSE:
            print(f"    template: {tpl_id or '—'}  (title read: {ev['title_read']!r}; from {ev['detected_from']}; "
                  f"override {ev['override']})" + "".join(f"\n    ⚠ {w}" for w in ev["warnings"]))
        if tpl_id is None:
            meta.update(status="unrecognised", issues=["form not recognised - set TEMPLATE_OVERRIDE for this file"])
            return dict(form_type=None, header=header, _meta=meta), ctx
        tpl = TEMPLATES[tpl_id]
        if tpl["kind"] == "record":
            data, issues, warnings = extract_record(ctx, img, raw_gray, tpl)
        elif tpl["kind"] == "matrix":
            data, issues, warnings = extract_mis(ctx, img, header, variant)
        else:
            data, issues, warnings = extract_stpass(ctx, img, raw_gray, tpl)
        hdr = build_header(tpl_id, header, ev.get("matched_phrase"))
        if tpl["kind"] == "stpass":
            hdr["days_printed"] = data.pop("days_printed")
        result = dict(form_type=tpl_id)
        if tpl["kind"] == "matrix":
            result["mis_variant"] = data.pop("mis_variant")
            meta["mis_variant_reason"] = data.pop("mis_variant_reason")
        result["header"] = hdr
        result.update(data)
        issues = ev["warnings"] + issues
        status = "not_run" if DRY_RUN else ("auto_ok" if not issues else "review")
        meta.update(status=status, issues=issues, warnings=warnings, **ctx.extra)
        result["_meta"] = meta
        return result, ctx
    except Exception as e:
        meta.update(status="failed", issues=[f"{type(e).__name__}: {e}"], traceback=traceback.format_exc(limit=-4))
        print("    ✗", meta["issues"][0])
        return dict(form_type=None, _meta=meta), ctx
    finally:
        meta["seconds"] = round(time.time() - t0, 1)
        meta["model_calls"] = len(ctx.calls)
        meta["overlay"] = str(ctx.overlay) if getattr(ctx, "overlay", None) else None

def out_stem(pdf_rel):
    return re.sub(r"[^A-Za-z0-9/]+", "_", str(Path(pdf_rel).with_suffix(""))).strip("_")

def strip_meta(obj):
    if isinstance(obj, dict):
        return {k: strip_meta(v) for k, v in obj.items() if k != "_meta"}
    if isinstance(obj, list):
        return [strip_meta(x) for x in obj]
    return obj

def ocr_pdf(pdf_path, pdf_rel=None):
    pdf_path = Path(pdf_path)
    pdf_rel = pdf_rel or pdf_path.name
    doc = pymupdf.open(pdf_path)
    n = len(doc) if MAX_PAGES_PER_PDF is None else min(len(doc), MAX_PAGES_PER_PDF)
    print(f"▶ {pdf_rel}  ({len(doc)} page{'s' if len(doc) > 1 else ''})")
    pages = []
    for pno in range(n):
        page_dir = OUTPUT_DIR / "debug" / out_stem(pdf_rel) / f"p{pno + 1}"
        if page_dir.exists():
            shutil.rmtree(page_dir)
        res, ctx = process_page(pdf_path, pdf_rel, doc[pno], pno + 1, page_dir)
        m = res["_meta"]
        print(f"  p{pno + 1}: {res.get('form_type') or '—'}{':' + res['mis_variant'] if res.get('mis_variant') else ''} "
              f"· {m.get('status')} · {m.get('model_calls')} calls · {m.get('seconds')} s"
              + (f" · {len(m.get('issues', []))} issue(s)" if m.get("issues") else ""))
        for i in m.get("issues", [])[:6]:
            print("      ✗", i)
        for w in m.get("warnings", [])[:4]:
            print("      ⚠", w)
        pages.append(res)
    out = OUTPUT_DIR / "json" / f"{out_stem(pdf_rel)}.json"
    out.parent.mkdir(parents=True, exist_ok=True)
    payload = pages if INCLUDE_META_IN_JSON else strip_meta(pages)
    out.write_text(json.dumps(payload, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
    return pages

## 12. Run on all PDFs

In [ ]:
RESULTS = {}
t_all = time.time()
for pdf, rel in PDFS:
    RESULTS[rel] = ocr_pdf(pdf, rel)
print(f"\ndone: {len(RESULTS)} PDF(s) in {time.time() - t_all:.0f} s -> {OUTPUT_DIR / 'json'}")

summary = pd.DataFrame([dict(pdf=rel, page=p["_meta"]["page"], form_type=p.get("form_type"),
                             mis_variant=p.get("mis_variant"), status=p["_meta"].get("status"),
                             title_read=(p["_meta"].get("template_decision") or {}).get("title_read"),
                             decided_from=(p["_meta"].get("template_decision") or {}).get("detected_from"),
                             override=(p["_meta"].get("template_decision") or {}).get("override"),
                             n_entries=len(p.get("entries") or []),
                             n_mis_rows=sum(len(s["rows"]) for s in p.get("sections") or []),
                             n_parts=len(p.get("parts") or []),
                             issues=len(p["_meta"].get("issues", [])), calls=p["_meta"].get("model_calls"),
                             seconds=p["_meta"].get("seconds"))
                        for rel, pages in RESULTS.items() for p in pages])
summary.to_csv(OUTPUT_DIR / "pages_summary.csv", index=False, encoding="utf-8-sig")
summary

### Look at one result
`inspect_page(pdf_name_part, page)` shows the overlay, every model input with its raw answer, the issues and the JSON.

In [ ]:
def find_result(name_part, page=1):
    for rel, pages in RESULTS.items():
        if name_part.lower() in rel.lower():
            return rel, next((p for p in pages if p["_meta"]["page"] == page), None)
    return None, None

def inspect_page(name_part, page=1, max_calls=8, show_json=True):
    rel, res = find_result(name_part, page)
    if res is None:
        print("not found"); return
    m = res["_meta"]
    print(f"{rel} p{page}: {res.get('form_type')} {res.get('mis_variant') or ''} · status {m.get('status')}")
    print("template decision:", json.dumps(m.get("template_decision"), ensure_ascii=False))
    if m.get("overlay"):
        show(cv2.imread(m["overlay"]), "overlay", width=12)
    log = Path(m["debug_dir"]) / "model_io.jsonl"
    if log.exists():
        for line in log.read_text(encoding="utf-8").splitlines()[:max_calls]:
            c = json.loads(line)
            show(cv2.imread(c["image"]), f"{c['name']} · sent {c['image_wh']} · model view {c.get('model_view_wh')}", width=12)
            print("raw:", (c.get("raw") or "")[:1500], "\nerror:", c.get("error"))
    print("issues:", *m.get("issues", []), sep="\n  ")
    print("warnings:", *m.get("warnings", []), sep="\n  ")
    if show_json:
        print(json.dumps(strip_meta(res), indent=1, ensure_ascii=False)[:6000])

if RESULTS:
    inspect_page(next(iter(RESULTS)), 1, max_calls=3)

## 13. Tables for Excel / database

In [ ]:
rec_rows, mis_rows, st_rows = [], [], []
for rel, pages in RESULTS.items():
    for p in pages:
        base = dict(pdf=rel, page=p["_meta"]["page"], form_type=p.get("form_type"), status=p["_meta"].get("status"))
        hdr = p.get("header") or {}
        for i, e in enumerate(p.get("entries") or []):
            rec_rows.append(dict(base, month=hdr.get("month"), shop=hdr.get("shop"), entry=i, **e))
        for s in p.get("sections") or []:
            for kind, rows in (("row", s["rows"]), ("section_total", [s["section_total"]]), ("section_pct", [s["section_pct"]])):
                for r in rows:
                    if not r:
                        continue
                    for col in p.get("columns") or []:
                        mis_rows.append(dict(base, variant=p.get("mis_variant"), date=hdr.get("date"), section=s["section"],
                                             row_type=kind, sr_no=r.get("sr_no"), defect=r.get("defect"),
                                             reason=r.get("reason"), column=col, value=r.get(col)))
        for r in p.get("summary") or []:
            for col in p.get("columns") or []:
                mis_rows.append(dict(base, variant=p.get("mis_variant"), date=hdr.get("date"), section=None,
                                     row_type="summary", sr_no=None, defect=r.get("defect"), reason=None,
                                     column=col, value=r.get(col)))
        for part in p.get("parts") or []:
            for day, entry in part["days"].items():
                if entry == "NO PLAN":
                    st_rows.append(dict(base, sno=part["sno"], part_no=part["part_no"], day=day, shift=None,
                                        parameter=None, value="NO PLAN")); continue
                for sh, x in entry.items():
                    if x is None:
                        continue
                    if x == "NO PLAN":
                        st_rows.append(dict(base, sno=part["sno"], part_no=part["part_no"], day=day, shift=sh,
                                            parameter=None, value="NO PLAN")); continue
                    for k, val in x.items():
                        st_rows.append(dict(base, sno=part["sno"], part_no=part["part_no"], day=day, shift=sh,
                                            parameter=k, value=val))
with pd.ExcelWriter(OUTPUT_DIR / "results.xlsx") as xw:
    pd.DataFrame(summary).to_excel(xw, sheet_name="pages", index=False)
    rec = pd.DataFrame(rec_rows)
    for ft in (rec["form_type"].unique() if len(rec) else []):
        rec[rec["form_type"] == ft].dropna(axis=1, how="all").to_excel(xw, sheet_name=ft[:31], index=False)
    if mis_rows:
        pd.DataFrame(mis_rows).to_excel(xw, sheet_name="MIS_long", index=False)
    if st_rows:
        pd.DataFrame(st_rows).to_excel(xw, sheet_name="ST_PASS_long", index=False)
    review = [dict(pdf=rel, page=p["_meta"]["page"], form_type=p.get("form_type"), status=p["_meta"].get("status"),
                   issue=i) for rel, pages in RESULTS.items() for p in pages for i in p["_meta"].get("issues", [])]
    pd.DataFrame(review or [dict(issue="none")]).to_excel(xw, sheet_name="needs_review", index=False)
print("written:", OUTPUT_DIR / "results.xlsx", f"| entries {len(rec_rows)} · MIS cells {len(mis_rows)} · ST cells {len(st_rows)}")

## 14. Debug report
`report.html`: per page the template decision, overlay, every model input next to its raw answer, issues and JSON.
Images are linked relatively - download `ocr_all_forms_output.zip` (step 16), unzip and open `report.html`.

In [ ]:
def rel_link(path):
    return Path(os.path.relpath(path, OUTPUT_DIR)).as_posix()

def esc(x): return html.escape("" if x is None else str(x))

def build_report(results):
    css = ("body{font-family:system-ui,Segoe UI,sans-serif;margin:24px;background:#fafafa;color:#222}"
           "h2{margin-top:40px;border-bottom:2px solid #ccc}.page{background:#fff;border:1px solid #ddd;border-radius:8px;"
           "padding:12px;margin:14px 0}.auto_ok{border-left:6px solid #2e7d32}.review{border-left:6px solid #ef6c00}"
           ".failed,.unrecognised{border-left:6px solid #c62828}.not_run{border-left:6px solid #999}"
           "img{max-width:100%;border:1px solid #bbb}pre{background:#f3f3f3;padding:8px;white-space:pre-wrap;font-size:12px;"
           "max-height:420px;overflow:auto}.meta{color:#666;font-size:12px}.issue{color:#c62828}.warn{color:#ef6c00}"
           ".call{display:flex;gap:12px;margin:8px 0}.call>div{flex:1;min-width:0}")
    out = [f"<html><head><meta charset='utf-8'><title>OCR report</title><style>{css}</style></head><body>",
           f"<h1>OCR report – all forms</h1><p class='meta'>model {esc(MODEL_ID)} · dry run {DRY_RUN} · "
           f"{time.strftime('%Y-%m-%d %H:%M')}</p>"]
    for rel, pages in results.items():
        out.append(f"<h2>{esc(rel)}</h2>")
        for p in pages:
            m = p["_meta"]
            ev = m.get("template_decision") or {}
            out.append(f"<div class='page {esc(m.get('status'))}'><b>page {m['page']}</b> · <b>{esc(p.get('form_type'))}</b> "
                       f"{esc(p.get('mis_variant') or '')} · status <b>{esc(m.get('status'))}</b>"
                       f"<p class='meta'>title read: {esc(ev.get('title_read'))} · decided from {esc(ev.get('detected_from'))} "
                       f"· override {esc(ev.get('override'))} · deskew {esc(m.get('deskew_deg'))}° · "
                       f"{esc(m.get('model_calls'))} calls · {esc(m.get('seconds'))} s</p>")
            out += [f"<div class='issue'>✗ {esc(i)}</div>" for i in m.get("issues", [])]
            out += [f"<div class='warn'>⚠ {esc(w)}</div>" for w in m.get("warnings", [])]
            if m.get("overlay"):
                out.append(f"<details><summary>overlay</summary><img src='{rel_link(m['overlay'])}'></details>")
            log = Path(m["debug_dir"]) / "model_io.jsonl"
            if log.exists():
                out.append("<details><summary>model calls</summary>")
                for line in log.read_text(encoding="utf-8").splitlines():
                    c = json.loads(line)
                    out.append(f"<div class='call'><div><div class='meta'>{esc(c['name'])} · sent {esc(c['image_wh'])} · "
                               f"view {esc(c.get('model_view_wh'))} · tokens {esc(c.get('input_tokens'))}/"
                               f"{esc(c.get('output_tokens'))} · {esc(c.get('seconds'))} s · {esc(c.get('error') or 'ok')}</div>"
                               f"<img src='{rel_link(c['image'])}'></div><div><pre>{esc(c.get('raw'))}</pre>"
                               f"<details><summary>prompt</summary><pre>{esc(c.get('prompt'))}</pre></details></div></div>")
                out.append("</details>")
            out.append(f"<details><summary>JSON</summary><pre>{esc(json.dumps(strip_meta(p), indent=1, ensure_ascii=False))}"
                       f"</pre></details></div>")
    out.append("</body></html>")
    return "\n".join(out)

report_path = OUTPUT_DIR / "report.html"
report_path.write_text(build_report(RESULTS), encoding="utf-8")
print("report:", report_path, f"({report_path.stat().st_size / 1e6:.1f} MB)")

## 15. Accuracy against `OCR_JSON_SCHEMA.md` (optional)
The schema file holds hand-checked JSON for the template PDFs. Put it in `MyDrive/MeenakshiPublic/` (or next to the
notebook); PDFs with the same file name are scored on page 1. Integers / numbers must match exactly; text is compared
after normalising case and spaces (CER reported).

In [ ]:
SCHEMA_CANDIDATES = [Path(f"{DRIVE_ROOT}/OCR_JSON_SCHEMA.md"), Path("OCR_JSON_SCHEMA.md")]

def load_truth(path):
    truth, txt = {}, path.read_text(encoding="utf-8")
    for m in re.finditer(r"^## `([^`]+\.pdf)`.*?```json\s*(.*?)```", txt, re.S | re.M):
        try:
            truth[m.group(1).lower()] = json.loads(m.group(2))
        except json.JSONDecodeError as e:
            print("skip", m.group(1), e)
    return truth

def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cmp_val(want, got):
    if isinstance(want, (int, float)) and not isinstance(want, bool):
        return to_num(got) == want, None
    if want is None:
        return got in (None, ""), None
    a, b = norm_text(got), norm_text(want)
    return a == b, levenshtein(a, b) / max(len(b), 1)

def compare(truth, pred):
    rows = []
    def add(path, want, got):
        ok, cer = cmp_val(want, got)
        rows.append(dict(field=path, truth=want, pred=got, correct=ok, cer=cer))
    if "entries" in truth:
        pe = pred.get("entries") or []
        for i, te in enumerate(truth["entries"]):
            for k, v in te.items():
                add(f"entries[{i}].{k}", v, (pe[i] if i < len(pe) else {}).get(k))
    if "sections" in truth:
        def by_key(doc):
            idx = {}
            for s in doc.get("sections") or []:
                for r in s.get("rows") or []:
                    idx[("row", r.get("sr_no"), norm_text(r.get("defect")))] = r
            return idx
        pidx = by_key(pred)
        for (kind, sr, lab), tr in by_key(truth).items():
            pr = pidx.get((kind, sr, lab)) or next((r for (k2, s2, l2), r in pidx.items() if s2 == sr and sr is not None), {})
            for k, v in tr.items():
                if k in ("sr_no", "defect", "reason"):
                    continue
                add(f"row {sr} {lab}.{k}", v, pr.get(k))
    if "parts" in truth:
        pp = {p.get("sno"): p for p in pred.get("parts") or []}
        for tp in truth["parts"]:
            q = pp.get(tp["sno"], {})
            for day, te in tp["days"].items():
                qe = (q.get("days") or {}).get(day)
                if te == "NO PLAN" or qe == "NO PLAN":
                    add(f"part {tp['sno']} day {day}", te if te == "NO PLAN" else "data", qe if qe == "NO PLAN" else "data")
                    continue
                for sh, tx in te.items():
                    qx = (qe or {}).get(sh) if isinstance(qe, dict) else None
                    if not isinstance(tx, dict):
                        add(f"part {tp['sno']} day {day} {sh}", tx, qx if not isinstance(qx, dict) else "data")
                        continue
                    for k, v in tx.items():
                        add(f"part {tp['sno']} day {day} {sh}.{k}", v, qx.get(k) if isinstance(qx, dict) else None)
    for k, v in (truth.get("header") or {}).items():
        if k in ("company", "title", "days_printed"):
            continue
        add(f"header.{k}", v, (pred.get("header") or {}).get(k))
    return rows

schema_file = next((p for p in SCHEMA_CANDIDATES if p.exists()), None)
if schema_file is None or DRY_RUN:
    print("no OCR_JSON_SCHEMA.md found (or dry run) - accuracy skipped")
else:
    TRUTH = load_truth(schema_file)
    ev_rows = []
    for rel, pages in RESULTS.items():
        t = TRUTH.get(Path(rel).name.lower())
        if t and pages:
            for r in compare(t, pages[0]):
                ev_rows.append(dict(pdf=rel, form_type=pages[0].get("form_type"), **r))
    if ev_rows:
        ev = pd.DataFrame(ev_rows)
        ev.to_csv(OUTPUT_DIR / "evaluation.csv", index=False, encoding="utf-8-sig")
        display(ev.groupby(["pdf", "form_type"]).agg(fields=("correct", "size"), accuracy=("correct", "mean"),
                                                     mean_cer=("cer", "mean")))
        display(ev[~ev["correct"]].head(60))
    else:
        print("no PDF name in the run matches a schema entry")

## 16. One-off: OCR a single PDF (upload or Drive path)

In [ ]:
# path = "/content/drive/MyDrive/MeenakshiPublic/some_new_scan.pdf"
# pages = ocr_pdf(path)
# print(json.dumps(strip_meta(pages), indent=1, ensure_ascii=False)[:5000])
#
# upload from your computer:
# from google.colab import files
# for name in files.upload():
#     pages = ocr_pdf(name); print(json.dumps(strip_meta(pages), indent=1, ensure_ascii=False)[:5000])

In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print(f"zipped: {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB)")